# AF2 feature-frequency classification adapter — seed 42
Menjalankan static audit, kontrol `AF2FFA0`, kandidat `AF2FFA1`, lalu keputusan validation. Test tidak diekstrak. Output disimpan langsung ke satu folder proyek Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, json, os, shutil, subprocess, sys, tarfile, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/af2-feature-frequency-adapter'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
for module_name in list(sys.modules):
    if module_name=='coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root,require_project_artifact
REQ=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt')
PROJECT=resolve_drive_project_root(required_relative_paths=REQ)
ARCHIVE=require_project_artifact(PROJECT,REQ[0]); AF2=require_project_artifact(PROJECT,REQ[1])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'train/images').is_dir() and (DATA/'val/images').is_dir()
assert not (DATA/'test').exists(), 'STOP: test tersedia.'
GROUPED=DATA/'faruq_grouped_summary.json'; assert GROUPED.is_file(), GROUPED
OUTPUT=PROJECT/'experiments/faruq-v3-af2-feature-frequency-adapter-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
STATIC=OUTPUT/'static_audit.json'
print('GPU:',torch.cuda.get_device_name(0)); print('PROJECT:',PROJECT); print('OUTPUT:',OUTPUT)

In [ ]:
from coffee_detector.af2_ffa.audit import run_af2_ffa_static_audit
audit=run_af2_ffa_static_audit(AF2,STATIC,device='cuda:0')
print('PARAMETERS:',{'source':audit['source_parameters'],'candidate':audit['candidate_parameters'],'added':audit['added_parameters'],'fraction':audit['added_fraction']})
print('GATES:',audit['gates']); print('DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: static audit gagal; jangan training.'

In [ ]:
import time
def run_arm(arm):
    result=OUTPUT/'val_reports'/f'{arm}_seed42_result.json'; log=OUTPUT/f'{arm}_seed42_run.log'
    if result.is_file(): print('REUSE COMPLETE:',arm); return json.loads(result.read_text())
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffa_arm','--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
    print('START/RESUME:',arm,'| log=',log,flush=True)
    with log.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    seen=-1
    while process.poll() is None:
        lines=log.read_text(errors='replace').splitlines() if log.is_file() else []
        epochs=sum(1 for line in lines if line.lstrip().startswith(('1/','2/','3/','4/','5/','6/','7/','8/','9/')) and 'GPU_mem' not in line)
        if epochs!=seen: print(f'{arm}: status berubah; pantau {log}',flush=True); seen=epochs
        time.sleep(30)
    if process.returncode:
        print('\n'.join(log.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{arm} gagal: {process.returncode}')
    assert result.is_file(),result; return json.loads(result.read_text())
results={arm:run_arm(arm) for arm in ('AF2FFA0','AF2FFA1')}
print({arm:{k:v for k,v in item['metrics'].items() if k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')} for arm,item in results.items()})

In [ ]:
from coffee_detector.experiments.run_faruq_v3_af2_ffa_decision import run_faruq_v3_af2_ffa_decision
decision=run_faruq_v3_af2_ffa_decision(OUTPUT,seed=42)
print('DECISION:',decision['decision']); print('NEXT:',decision['next'])
print('Kirim values, candidate_minus_control, criteria, dan decision. Jangan buka test atau seed lain bila FAIL.')